In [1]:
audio_path = "dataset_gen/free_music/rotormotor/mp3s/001 Good Girl.mp3"  # ← Change to your file

import torch
from torchcodec.decoders import AudioDecoder
from IPython.display import Audio, display
import dac
import torchaudio.functional as F  # Only for resampling

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load DAC
model_path = dac.utils.download(model_type="24khz")
dac_model = dac.DAC.load(model_path).to(device).eval()
TARGET_SR = 24000

# %% [markdown]
# ### 1. Load Audio (torchcodec)
decoder = AudioDecoder(audio_path)
samples = decoder.get_all_samples()
waveform, sr = samples.data, samples.sample_rate
print(f"Loaded: {waveform.shape} @ {sr}Hz")

# %% [markdown]
# ### 2. Preprocess
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)  # [1, T]
if sr != TARGET_SR:
    waveform = F.resample(waveform, sr, TARGET_SR)

waveform = waveform.to(device)
print(f"Prepared: {waveform.shape} @ {TARGET_SR}Hz")

# %% [markdown]
# ### 3. Encode → Decode
with torch.no_grad():
    # Encode
    x = dac_model.preprocess(waveform, TARGET_SR)
    z, codes, _, _, _ = dac_model.encode(x, n_quantizers=16)
    print(f"Codes: {codes.shape}")  # [B, K, T]
    
    # Decode
    codes_dac = codes.permute(1, 0, 2)  # DAC expects [K, B, T]
    z_dec = dac_model.quantizer.from_codes(codes_dac)
    z_dec = z_dec[0] if isinstance(z_dec, tuple) else z_dec
    decoded = dac_model.decode(z_dec)
    
    # Fix DAC's dimension swap ([K, B, S] → [B, K, S])
    if decoded.dim() == 3 and decoded.shape[0] != codes.shape[0]:
        decoded = decoded.transpose(0, 1)
    if decoded.shape[1] > 1:
        decoded = decoded.mean(dim=1, keepdim=True)

print(f"Decoded: {decoded.shape}")

# %% [markdown]
# ### 4. Play & Verify
print("\n🔊 Original:")
display(Audio(waveform.cpu().numpy(), rate=TARGET_SR))

print("\n🔊 Reconstructed:")
display(Audio(decoded.cpu().numpy(), rate=TARGET_SR))

if waveform.shape == decoded.shape:
    snr = 10 * torch.log10(torch.mean(waveform**2) / torch.mean((waveform - decoded)**2 + 1e-8))
    print(f"\n✅ SNR: {snr.item():.2f} dB")

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cu130) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent cal